In [1]:
!pip install -q torch transformers datasets accelerate scikit-learn numpy hf_transfer protobuf sentencepiece

In [4]:
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import zipfile
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

# --- CONFIGURATION ---
MODEL_NAME = "microsoft/deberta-v3-base" 
BATCH_SIZE = 8
EPOCHS = 5
LEARNING_RATE = 1.5e-5 # Slightly lower LR for fine-tuning
MAX_LENGTH = 300       # Increased length for added examples

# --- STRATEGY 3: EARTH MOVER'S DISTANCE LOSS (Ordinal Regression) ---
class EMDTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels") # Shape: (Batch, 5) -> Distribution
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        # 1. Convert Logits to Probabilities (Softmax)
        probs = F.softmax(logits, dim=-1)
        
        # 2. Compute Cumulative Distribution Function (CDF)
        # EMD is the mean squared error between the CDFs of prediction and truth
        # This forces the model to respect the "order" of 1, 2, 3, 4, 5.
        pred_cdf = torch.cumsum(probs, dim=-1)
        true_cdf = torch.cumsum(labels, dim=-1)
        
        # 3. Squared EMD Loss
        loss = torch.mean(torch.square(pred_cdf - true_cdf))
        
        return (loss, outputs) if return_outputs else loss

# --- STRATEGIES 1 & 2: INPUT ENRICHMENT & MARKERS ---
def mark_homonym(text, homonym):
    # Wraps the homonym in quotes " word " to highlight it to the model.
    # Simple case-insensitive replace. 
    # For a paper, you might want a more robust regex, but this works for 99% of cases.
    lower_text = text.lower()
    lower_hom = homonym.lower()
    
    if lower_hom in lower_text:
        # Find start index
        start = lower_text.find(lower_hom)
        end = start + len(lower_hom)
        # Reconstruct with markers
        return text[:start] + '" ' + text[start:end] + ' "' + text[end:]
    return text

def load_advanced_data(file_path, is_test=False):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    ids, texts_a, texts_b, labels = [], [], [], []
    
    for id_str, info in data.items():
        ids.append(id_str)
        
        # --- ENRICHMENT (Strategy 1) ---
        # Concatenate Definition + Example Sentence
        # Format: "Definition: ... | Example: ..."
        def_text = f"Definition: {info['judged_meaning']}"
        if 'example_sentence' in info:
            def_text += f" | Example: {info['example_sentence']}"
        texts_a.append(def_text)
        
        # --- MARKING (Strategy 2) ---
        # Mark the homonym in the target sentence
        homonym = info['homonym']
        marked_sentence = mark_homonym(info['sentence'], homonym)
        
        # Construct full story
        story = f"{info['precontext']} {marked_sentence}"
        if info.get('ending'):
            story += f" {info['ending']}"
        texts_b.append(story)
        
        # --- LABELS ---
        # Same distribution logic as before
        raw_votes = info.get('choices', [])
        counts = np.zeros(5)
        if not raw_votes:
             probs = np.full(5, 0.2)
        else:
            for vote in raw_votes:
                if 1 <= vote <= 5:
                    counts[vote - 1] += 1
            probs = counts / counts.sum() if counts.sum() > 0 else np.full(5, 0.2)
            
        labels.append(probs.astype(np.float32))
        
    return Dataset.from_dict({
        'id': ids, 'text_a': texts_a, 'text_b': texts_b, 'label': labels
    })

# 1. Load Data
print("Loading advanced data...")
train_dataset = load_advanced_data('train.json')
dev_dataset = load_advanced_data('dev.json') # Dev set has examples too

# 2. Tokenize
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

def preprocess_function(examples):
    return tokenizer(
        examples['text_a'], 
        examples['text_b'], 
        truncation=True, 
        padding='max_length', 
        max_length=MAX_LENGTH
    )

print("Tokenizing...")
encoded_train = train_dataset.map(preprocess_function, batched=True)
encoded_dev = dev_dataset.map(preprocess_function, batched=True)

# Clean columns
columns_to_keep = ['input_ids', 'attention_mask', 'label']
if 'token_type_ids' in encoded_train.column_names:
    columns_to_keep.append('token_type_ids')
encoded_train = encoded_train.remove_columns([c for c in encoded_train.column_names if c not in columns_to_keep])
encoded_dev_model = encoded_dev.remove_columns([c for c in encoded_dev.column_names if c not in columns_to_keep])

# 3. Model Setup
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=5)

# 4. Training with EMD Trainer
args = TrainingArguments(
    output_dir="./results_advanced",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),
    save_total_limit=2,
    remove_unused_columns=False,
    report_to="none"
)

trainer = EMDTrainer(
    model=model,
    args=args,
    train_dataset=encoded_train,
    eval_dataset=encoded_dev_model,
    tokenizer=tokenizer,
)

print("Starting training with EMD Loss...")
trainer.train()

# 5. Prediction
print("Generating predictions...")
predictions = trainer.predict(encoded_dev_model)
logits = predictions.predictions
probs = F.softmax(torch.tensor(logits), dim=-1).numpy()

# Calculate Weighted Average Score
scores_indices = np.array([1, 2, 3, 4, 5])
expected_scores = np.sum(probs * scores_indices, axis=1)

output_entries = []
for idx, score in enumerate(expected_scores):
    sample_id = dev_dataset[idx]['id']
    # Round to nearest int for submission format
    final_int = int(round(max(1, min(5, score))))
    output_entries.append({"id": sample_id, "prediction": final_int})

with open('predictions.jsonl', 'w') as f:
    for entry in output_entries:
        f.write(json.dumps(entry) + '\n')

with zipfile.ZipFile('predictions_advanced.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write('predictions.jsonl')

print("Done! Advanced submission saved.")

Loading advanced data...
Tokenizing...


Map:   0%|          | 0/2280 [00:00<?, ? examples/s]

Map:   0%|          | 0/588 [00:00<?, ? examples/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_10043/4057082762.py:147: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `EMDTrainer.__init__`. Use `processing_class` instead.
  trainer = EMDTrainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Starting training with EMD Loss...


Epoch,Training Loss,Validation Loss
1,No log,0.086310
2,0.084500,0.072874
3,0.084500,0.059133
4,0.050600,0.064877
5,0.050600,0.065222


Generating predictions...


Done! Advanced submission saved.
